# 💧 Generate Antidoom FTPO Dataset (LFM Base via llama.cpp)

This notebook generates an FTPO dataset for [Antidoom](https://github.com/Liquid4All/antidoom) training using **LFM2.5-1.2B-Base** served by a local llama.cpp server.

## Setup

1. Download the GGUF: `hf download LiquidAI/LFM2.5-1.2B-Base-GGUF --local-dir models/lfm-base`
2. Start llama.cpp: `llama-server -m models/lfm-base/LFM2.5-1.2B-Base-BF16.gguf --port 8080 -ngl 99 --temp 0.01 --top-k 50 --top-p 1.0 --min-p 0.01`
3. Run this notebook

In [9]:
!pip install openai datasets huggingface_hub tqdm transformers python-dotenv

  Using cached python_dotenv-1.2.3-py3-none-any.whl.metadata (29 kB)
Using cached python_dotenv-1.2.3-py3-none-any.whl (22 kB)

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [10]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [11]:
import time
from openai import OpenAI

# Connect to local llama.cpp server
client = OpenAI(base_url="http://localhost:8080/v1", api_key="not-needed")

# Verify connection
models = client.models.list()
print(f"Connected to server. Available models: {[m.id for m in models.data]}")

Connected to server. Available models: ['models/lfm-base/LFM2.5-1.2B-Base-BF16.gguf']


# 🔍 Doom Loop Detection

In [12]:
from dataclasses import dataclass
from tqdm.auto import tqdm


@dataclass(frozen=True)
class RepeatHit:
    start: int
    end: int
    period: int
    repeats: int
    snippet: str

    @property
    def repeat_start(self):
        return self.start + self.period


def _verify_repetition_at(text, start_pos, period, min_repeats, min_total_repeated):
    if period < 1 or start_pos < 0 or start_pos + period > len(text):
        return False, None
    pattern = text[start_pos : start_pos + period]
    reps = 0
    pos = start_pos
    while pos + period <= len(text) and text[pos : pos + period] == pattern:
        reps += 1
        pos += period
    end_pos = pos
    pos = start_pos - period
    while pos >= 0 and text[pos : pos + period] == pattern:
        reps += 1
        start_pos = pos
        pos -= period
    total = reps * period
    if reps >= min_repeats and total >= min_total_repeated:
        snippet = pattern if len(pattern) <= 100 else pattern[:100] + "..."
        return True, RepeatHit(start_pos, end_pos, period, reps, snippet)
    return False, None


def find_inner_repetition(
    text,
    min_repeats=4,
    max_period=1024,
    min_period=1,
    min_total_repeated=60,
    sample_len=16,
    sample_interval=128,
):
    if not text or len(text) < min_total_repeated:
        return False, None
    n = len(text)
    for sample_pos in range(0, n - sample_len, sample_interval):
        fingerprint = text[sample_pos : sample_pos + sample_len]
        for other_pos in [
            text.find(fingerprint, sample_pos + sample_len),
            text.rfind(fingerprint, 0, sample_pos),
        ]:
            if other_pos == -1:
                continue
            candidate_period = abs(other_pos - sample_pos)
            if min_period <= candidate_period <= max_period:
                found, hit = _verify_repetition_at(
                    text, min(sample_pos, other_pos), candidate_period,
                    min_repeats=min_repeats, min_total_repeated=min_total_repeated,
                )
                if found:
                    return True, hit
    return False, None

# 📥 Load Prompts

In [13]:
from datasets import load_dataset

dataset = load_dataset("LiquidAI/antidoom-mix-v1.0", split="train")
dataset = dataset.shuffle(seed=42)
print(f"📥 Loaded {len(dataset)} prompts")

PROMPT_TEMPLATE = (
    "{prompt}\n\n"
    'Think through the problem step by step, then respond with your final answer as "Answer: <your answer>".'
)

📥 Loaded 478229 prompts


# 🚀 Generate & Detect Doom Loops

Using the OpenAI-compatible API served by llama.cpp with LFM2.5-1.2B-Base. Sampling parameters match the [Antidoom default config](https://github.com/Liquid4All/antidoom/blob/main/src/antidoom/config.py).

The Base model produces ~42% doom loops on the full prompt mix — no source filtering needed.

In [ ]:
NUM_PROMPTS = 10
MAX_NEW_TOKENS = 4000
TEMPERATURE = 0.01

prompts = dataset.select(range(min(NUM_PROMPTS, len(dataset))))

doom_results = []
clean_results = []

start_time = time.time()

for sample in tqdm(prompts, desc="Generating"):
    prompt_text = sample["conversations"][0]["value"]
    formatted = PROMPT_TEMPLATE.format(prompt=prompt_text)

    response = client.completions.create(
        model="default",
        prompt=formatted,
        max_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_p=1.0,
        extra_body={"min_p": 0.01, "top_k": 50},
    )

    generated_text = response.choices[0].text
    found, hit = find_inner_repetition(generated_text)

    result = {
        "prompt_text": prompt_text,
        "full_prompt": formatted,
        "generated_text": generated_text,
        "generated_length": response.usage.completion_tokens,
        "hit": hit,
    }
    (doom_results if found else clean_results).append(result)

elapsed = time.time() - start_time
total = len(doom_results) + len(clean_results)
doom_rate = len(doom_results) / total if total > 0 else 0

print(f"\n📊 Generation Results:")
print(f"   Total completions: {total}")
print(f"   🔄 Doom loops: {len(doom_results)} ({doom_rate:.1%})")
print(f"   ✅ Clean: {len(clean_results)} ({1 - doom_rate:.1%})")
print(f"   ⏱️  Time: {elapsed:.1f}s ({elapsed/total:.1f}s per sample)")

if doom_results:
    ex = doom_results[0]
    print(f"\n📝 Example doom loop:")
    print(f'   Pattern: "{ex["hit"].snippet}"')
    print(f"   Repeats: {ex['hit'].repeats}x ({ex['hit'].period} chars per repeat)")

# 🎯 Extract FTPO Training Pairs

FTPO needs single-token preference pairs at each doom loop's start position. For each doom loop we:
1. **Locate** the token where the repetition begins (the "rejected" token)
2. **Query** the llama.cpp server for top-50 token probabilities at that position
3. **Extract** the top-20 alternative tokens as "chosen" candidates

We load only the HuggingFace **tokenizer** (no model weights) for tokenization, and reuse the already-running llama.cpp server for logprobs via its native `/completion` endpoint with `n_probs`.

In [15]:
from transformers import AutoTokenizer

model_id = "LiquidAI/LFM2.5-1.2B-Base"

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"✅ Tokenizer loaded ({model_id})")

✅ Tokenizer loaded (LiquidAI/LFM2.5-1.2B-Base)


In [16]:
import requests

LLAMA_URL = "http://localhost:8080/completion"


def find_rejected_token_index(generated_ids, hit, tokenizer):
    """Map the doom loop's repeat_start character position to a token index."""
    gen_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    char_pos = hit.repeat_start
    while char_pos < len(gen_text) and gen_text[char_pos].isspace():
        char_pos += 1
    if char_pos >= len(gen_text):
        return None
    for i in range(len(generated_ids)):
        prefix = tokenizer.decode(generated_ids[: i + 1], skip_special_tokens=True)
        if len(prefix) > char_pos:
            return i
    return None


ftpo_pairs = []

for result in tqdm(doom_results, desc="Extracting FTPO pairs"):
    generated_ids = tokenizer.encode(result["generated_text"], add_special_tokens=False)

    reject_idx = find_rejected_token_index(generated_ids, result["hit"], tokenizer)
    if reject_idx is None or reject_idx == 0:
        continue

    input_ids = tokenizer.encode(result["full_prompt"], add_special_tokens=False)
    prompt_ids = input_ids + generated_ids[:reject_idx]
    rejected_id = generated_ids[reject_idx]

    # Get top-50 token probabilities at the rejection position via llama.cpp
    context_text = tokenizer.decode(prompt_ids, skip_special_tokens=True)
    resp = requests.post(LLAMA_URL, json={
        "prompt": context_text,
        "n_predict": 1,
        "n_probs": 50,
        "temperature": 0.0,
    })
    data = resp.json()

    if "completion_probabilities" not in data or not data["completion_probabilities"]:
        continue

    top_tokens = data["completion_probabilities"][0]["top_logprobs"]
    chosen_ids = []
    for t in top_tokens:
        tid = t["id"]
        if tid == rejected_id:
            continue
        if len(tokenizer.decode([tid]).strip()) >= 1:
            chosen_ids.append(tid)
        if len(chosen_ids) >= 20:
            break

    if not chosen_ids:
        continue

    ftpo_pairs.append({
        "full_prompt": result["full_prompt"],
        "prompt_ids": prompt_ids,
        "chosen_ids": chosen_ids,
        "rejected_token_id": rejected_id,
    })

print(f"\n✅ Extracted {len(ftpo_pairs)} FTPO training pairs from {len(doom_results)} doom loops")
if ftpo_pairs:
    ex = ftpo_pairs[0]
    rej_text = tokenizer.decode([ex["rejected_token_id"]])
    cho_texts = [tokenizer.decode([c]) for c in ex["chosen_ids"][:5]]
    print(f"\n📝 Example pair:")
    print(f'   ❌ Rejected token: "{rej_text}"')
    print(f"   ✅ Top chosen alternatives: {cho_texts}")
    print(f"   📏 Context length: {len(ex['prompt_ids'])} tokens")

Extracting FTPO pairs: 100%|██████████| 2/2 [00:00<00:00, 24.47it/s]


✅ Extracted 2 FTPO training pairs from 2 doom loops

📝 Example pair:
   ❌ Rejected token: " one"
   ✅ Top chosen alternatives: [' the', ' client', ' either', ' user', ' a']
   📏 Context length: 234 tokens


# 📤 Push Dataset to Hugging Face Hub

Push the FTPO training pairs as a Hugging Face dataset. Each row contains:
- `full_prompt` — the formatted prompt text (used for evaluation after training)
- `prompt_ids` — token IDs of the context up to the rejection position
- `chosen_ids` — top-20 alternative token IDs (the "good" continuations)
- `rejected_token_id` — the single token that starts the doom loop

In [17]:
from datasets import Dataset as HFDataset

FTPO_DATASET_REPO = "iamleonie/antidoom-test"

ftpo_dataset = HFDataset.from_list(ftpo_pairs)
ftpo_dataset.push_to_hub(FTPO_DATASET_REPO)

print(f"✅ Pushed {len(ftpo_dataset)} FTPO pairs to {FTPO_DATASET_REPO}")
print(f"📏 Columns: {ftpo_dataset.column_names}")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 220.02ba/s]
Processing Files (1 / 1): 100%|██████████| 7.56kB / 7.56kB,   724B/s  
New Data Upload: 100%|██████████| 7.56kB / 7.56kB,   724B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.37s/ shards]


✅ Pushed 2 FTPO pairs to iamleonie/antidoom-test
📏 Columns: ['full_prompt', 'prompt_ids', 'chosen_ids', 'rejected_token_id']


In [18]:
from datasets import load_dataset as _reload

ds = _reload(FTPO_DATASET_REPO, split="train")
print(f"📊 Sanity check — reloaded {len(ds)} rows from {FTPO_DATASET_REPO}")
print(f"📏 Columns: {ds.column_names}")

if len(ds) > 0:
    ex = ds[0]
    print(f"\n📝 First example:")
    print(f"   prompt_ids length: {len(ex['prompt_ids'])}")
    print(f"   chosen_ids: {ex['chosen_ids'][:5]}...")
    print(f"   rejected_token_id: {ex['rejected_token_id']}")
    rej = tokenizer.decode([ex["rejected_token_id"]])
    cho = [tokenizer.decode([c]) for c in ex["chosen_ids"][:5]]
    print(f'   ❌ Rejected: "{rej}"')
    print(f"   ✅ Chosen: {cho}")

Generating train split: 100%|██████████| 2/2 [00:00<00:00, 157.07 examples/s]

📊 Sanity check — reloaded 2 rows from iamleonie/antidoom-test
📏 Columns: ['full_prompt', 'prompt_ids', 'chosen_ids', 'rejected_token_id']

📝 First example:
   prompt_ids length: 234
   chosen_ids: [779, 7824, 4021, 5196, 768]...
   rejected_token_id: 1235
   ❌ Rejected: " one"
   ✅ Chosen: [' the', ' client', ' either', ' user', ' a']
